## 36122 Python Programming
### (Autumn 2026)
Group Project
----------------------------------

**Student Name and ID:**

Leon Hsu - 26324145

Ishani Bondade - 26147280

Chaemin Jin - 26492722

Niki Miyake - 14742605

In [ ]:
import os
from dotenv import load_dotenv

# API KEY from .env file
load_dotenv()

api_key = os.getenv("API_KEY")
print(api_key) 

9e0c3e5c285e403aac0964ce54a13fd6


**Step 1: Import required libraries**

In [2]:
%pip install requests beautifulsoup4 matplotlib

import requests
from bs4 import BeautifulSoup
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import webbrowser

API_KEY = api_key

Note: you may need to restart the kernel to use updated packages.


**Step 2: Web Scraping**

In [3]:
class NewsScrapper:
    def __init__(self, key):
        self.key = key
        self.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    def fetch_news(self, category):
        url = "https://newsapi.org/v2/top-headlines"

        #Request for API
        params = {
            'apikey': self.key,
            'category': category,
            'language': 'en',
            'pageSize': 10
        }

        try:
            response = requests.get(url, params=params, headers=self.headers)
            response.raise_for_status()
            articles = response.json().get('articles', [])

            results = []

            for article in articles:
                #Extracting the information from article
                scrapped_text = self.scrape_article(article['url'])

                #Store the processed news article
                results.append({
                    'title': article['title'],
                    'source': article['source']['name'],
                    'url': article['url'],
                    'content': scrapped_text
                })

            return results

        except Exception as e:
            return f"API ERROR!: {str(e)}"

    def scrape_article(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            # Extract the full text from the article (this is a simple example, you might need to adjust the selectors based on the website's structure)
            paragraphs = soup.find_all('p')
            text = ' '.join([p.get_text() for p in paragraphs])
            return text if text.strip() else "Could not extract full content!"
        except:
            return "Scraping failed for this url!"

**Step 3: GUI**

In [4]:
class NewsApp:
    def __init__(self, root, news_engine):
        self.root = root
        self.news_engine = news_engine
        self.root.title("News Scrapper")
        self.root.geometry("800x600")

        #Create the Menu
        tk.Label(root, text="Select the news category: ", font=("Times New Roman", 14, "bold")).pack(pady=10)
        self.category_list = ['Business', 'Entertainment', 'General', 'Health', 'Science', 'Sports', 'Technology']
        self.dropdown = ttk.Combobox(root, values=self.category_list, state="readonly")
        self.dropdown.set("General")
        self.dropdown.pack()

        #Create the button to fetch news
        tk.Button(root, text="Search News", command=self.handle_search, bg="blue", fg="white").pack(pady=10)

        #Display
        self.display = tk.Text(root, wrap=tk.WORD, padx=10, pady=10)
        self.display.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)

        self.display.config(state="disabled")

    def handle_search(self):
        selected_category = self.dropdown.get()
        self.display.config(state="normal")
        self.display.delete("1.0", tk.END)
        self.display.insert(tk.END, "Loading...\n")
        self.root.update()

        self.current_article = self.news_engine.fetch_news(selected_category)

        self.display.delete("1.0", tk.END)

        if isinstance(self.current_article, str):
            messagebox.showerror("Error", self.current_article)

        else:
            for i, news in enumerate(self.current_article):
                tag_name = f"article_{i}"
                self.display.insert(tk.END, f"Title: {news['title']}\n", tag_name, "bold")
                self.display.insert(tk.END, f"Source: {news['source']}\n")
                self.display.insert(tk.END, f"Content: {news['content'][:300]}...(Read full article.....)\n\n")
                self.display.insert(tk.END, f"URL: {news['url']}\n\n")
                self.display.insert(tk.END, "="*50 + "\n\n")

                #Clickable title
                self.display.tag_bind(tag_name, "<Double-1>", lambda e, idx=i: self.show_full_article(idx))
                self.display.tag_bind(tag_name, "<Enter>", lambda e: self.display.config(cursor="hand2"))
                self.display.tag_bind(tag_name, "<Leave>", lambda e: self.display.config(cursor=""))

                self.display.tag_config(tag_name, foreground="blue", underline=True, font=("Times New Roman", 12, "bold"))
                self.display.insert(tk.END, "="*50 + "\n\n")

        self.display.config(state="disabled")

    def show_full_article(self, index):
        #New pop up window to display the complete article
        article = self.current_article[index]
        new_window = tk.Toplevel(self.root)
        new_window.title(article['title'])
        new_window.geometry("600x400")

        #Header Information
        tk.Label(new_window, text=article['title'], font=("Times New Roman", 16, "bold"), wraplength=600, justify="left").pack(pady=10, padx=20)

        #Open article in browser if the article is not scrapped
        tk.Button(new_window, text="Open in Browser", bg="orange", command=lambda: webbrowser.open(article['url'])).pack(pady=5)

        #Make it scrollable
        text = tk.Text(new_window, wrap=tk.WORD, padx=20, pady=25, font=("Times New Roman", 12))
        text.pack(fill=tk.BOTH, expand=True)

        full_content = f"Source: {article['source']}\nURL: {article['url']}\n\n{article['content']}"
        text.insert("1.0", full_content)
        text.config(state="disabled") #This makes the text read-only

**Step 4: Launch the NEWS APP**

In [5]:
if __name__ == "__main__":
    try:
        main = tk.Tk()
        news = NewsScrapper(API_KEY)
        app = NewsApp(main, news)
        main.mainloop()

    except Exception as e:
        print(f"App closed. Error: {e}")